# 🔌 Connect to Your RADKit Service

![RADKit version](https://img.shields.io/badge/RADKit-1.9.6-blue?logo=cisco&logoColor=white) ![Python version](https://img.shields.io/badge/Python-3.12%2B-purple?logo=python&logoColor=white)

This lab focuses on one goal: **establishing a working connection to your RADKit service** using different authentication paths.

## What you will do
- Load local configuration from your `.env` file
- Connect through Cisco Cloud with SSO (most common)
- Connect with certificate-based login (automation friendly)
- Connect directly to your server without cloud access

---

## 1) Load Local Configuration

Before connecting, load environment variables from your `.env` file.

Expected values:
- `RADKIT_USER`: your CCO user ID
- `RADKIT_SERVICE`: your RADKit service ID (used by cloud-based connection methods)

In [4]:
import os
from dotenv import load_dotenv

load_dotenv()

True

---

## 2) Connect via Cisco Cloud (SSO Login)

**Best for:** Most users in interactive environments.

**Why choose this:** You authenticate with your Cisco account in a browser, then connect to your service by Service ID. Your machine does not need direct network access to the RADKit server.

**Flow:**
1. Create a `Client`.
2. Run `sso_login(...)` and complete browser authentication.
3. Resolve the target cloud service using `RADKIT_SERVICE`.

In [2]:
from radkit_client import Client
from radkit_client.sync import ClientStatus, ServiceStatus # Nice enums to get the status of the client and service

user_id = os.getenv("RADKIT_USER")
service_id = os.getenv("RADKIT_SERVICE")

with Client.create() as client:
    client.sso_login(user_id)
    print("✅ Authentication successful!") if client.status == ClientStatus.CONNECTED else print("🔥 Connection failed ...")
    
    service = client.service_cloud(service_id).wait()
    print(f"✅ Connection successful to service {service.service_id}!") if service.status == ServiceStatus.READY else print(f"🔥 Connection to service {service.service_id} failed ...")

<frozen radkit_common.utils.ssl>:518: CryptographyDeprecationWarning: Parsed a serial number which wasn't positive (i.e., it was negative or zero), which is disallowed by RFC 5280. Loading this certificate will cause an exception in a future release of cryptography.



A browser window was opened to continue the authentication process. Please follow the instructions there.

Authentication result received.
✅ Authentication successful!
✅ Connection successful to service 21km-e0xp-fcib!


---

## 3) Certificate Login (`certificate_login`)

**Best for:** Non-interactive workflows such as CI/CD, scheduled jobs, or headless scripts.

**Why choose this:** No browser prompt is required during login. Authentication uses locally stored client certificates.

**Important prerequisite:** You must enroll this machine/client first so certificate files exist.

---

Run the enrollment step once from a terminal:

```bash
python src/enroll-client.py
```

During enrollment, save your certificate private-key password. You will be prompted for it in the next cell.

After enrollment completes, run the next code cell.

In [ ]:
from radkit_client import Client
from radkit_client.sync import ClientStatus, ServiceStatus
import getpass

user_id = os.getenv("RADKIT_USER")
service_id = os.getenv("RADKIT_SERVICE")
private_key_password = getpass.getpass("🔑 Enter the password for your private key: ")

with Client.create() as client:
    client.certificate_login(identity=user_id, private_key_password=private_key_password)
    print("✅ Authentication successful!") if client.status == ClientStatus.CONNECTED else print("🔥 Connection failed ...")
    
    service = client.service_cloud(service_id).wait()
    print(f"✅ Connection successful to service {service.service_id}!") if service.status == ServiceStatus.READY else print(f"🔥 Connection to service {service.service_id} failed ...")

✅ Authentication successful!
✅ Connection successful to service 21km-e0xp-fcib!


---

## 4) Connect Directly (No Cloud)

**Best for:** Air-gapped environments, restricted outbound internet, or private-network-only deployments.

**Why choose this:** The client connects directly to the RADKit server endpoint over LAN/VPN, bypassing Cisco Cloud.

**You will need:**
- Your **CCO user ID**
- Your **E2EE validation token** (used as the password for direct auth)
- The server **hostname or IP address**
- The server **RPC port** (default: `8181`)

In [ ]:
from radkit_client import Client
from radkit_common.rpc.client_transports.verify import RPCVerificationError # Specific exception for failed RPC verification
import getpass

user_id = input("👤 Enter your CCO user id: ")
e2ee_validation_token = getpass.getpass("🔑 Enter your E2EE validation token: ")
server_address = input("🌐 Enter the server hostname or IP (e.g., radkit.lab.local or 10.10.10.20): ")
rpc_port = input("🔌 Enter the RPC port of your server (default is 8181): ")

with Client.create() as client:
    service = client.service_direct(
        username=user_id,
        host=server_address,
        port=int(rpc_port) if rpc_port else 8181,
        password=e2ee_validation_token
    )
    try:
        service.wait()
        print(f"✅ Connection successful to service!")
    except RPCVerificationError as e:
        print(f"🔥 Connection to service failed: {e}")
    except Exception as e:
        print(f"🔥 An unexpected error occurred while connecting to service: {e}")